# 10 — L'orizzonte mensile: la macro conta alla sua frequenza naturale? (Fase 4 bis)

**Motivazione**: tutti i test di Fase 4/5 erano a frequenza **daily** e non hanno
trovato edge dalla macro. Ma la EDA di Fase 1 aveva trovato il segnale macro a
frequenza **mensile** (`BTC vs CPI YoY = -0.40` contemporaneo, ~0 a daily). È
l'ultimo angolo di ricerca: **testare la macro alla sua frequenza naturale**.

Due domande:
1. **Lead/lag**: esiste una correlazione *predittiva* (lead) tra macro del mese
   `t` e rendimento del mese `t+1`?
2. **Modello**: un classificatore direzionale mensile guadagna accuracy OOS
   aggiungendo la macro?

## ⚠️ Il limite strutturale (da leggere PRIMA)
8 anni di crypto = **~100 mesi**. Dopo lag e warm-up restano ~80-90 osservazioni,
di cui ~45 out-of-sample. **È un campione minuscolo per un modello direzionale.**
Qualsiasi accuracy mensile va presa come *indicativa*, non conclusiva: con n~45,
l'errore standard sull'accuracy è ~7 punti percentuali. Questo notebook è onesto
su questo limite — non è risolvibile con più codice, solo con più tempo (o più
asset).

## Ipotesi — scritte PRIMA dei risultati
1. **H1 (esiste un lead macro debole)**: mi aspetto `corr(cpi_yoy[t],
   ret[t+1])` **negativa** (inflazione su → Fed hawkish → crypto giù), coerente
   col -0.40 contemporaneo di Fase 1, ma più debole perché predittiva.
2. **H2 (il modello mensile non ha edge robusto)**: col campione minuscolo, un
   classificatore direzionale non mostrerà un edge OOS *affidabile*, anche se la
   macro porta un filo di segnale.
3. **H3 (il limite è il campione, non l'assenza di segnale)**: la conclusione
   onesta sarà "c'è un indizio di segnale macro mensile, ma n è troppo piccolo
   per costruirci sopra".


In [1]:
import numpy as np
import pandas as pd

from src.assets.asset import get_asset_by_symbol
from src.ingestion.tier1.yahoo_finance import YahooFinanceSource
from src.features.macro_features import build_macro_features
from src.features.news_features import lead_lag_table
from src.models.multifactor import fit_predict_walk_forward
from pathlib import Path


## 1. Dati mensili: BTC end-of-month + macro point-in-time-safe

In [2]:
src = YahooFinanceSource()
ohlcv = src.fetch_ohlcv(get_asset_by_symbol('BTC'), start='2018-01-01', interval='1d').sort_index()
close_m = ohlcv['close'].resample('ME').last()
ret_m = close_m.pct_change()
print('monthly bars:', len(close_m), '|', str(close_m.index.min())[:7], '->', str(close_m.index.max())[:7])

# monthly technical features (causal)
feat = pd.DataFrame(index=close_m.index)
feat['ret_1m'] = ret_m
feat['mom_3m'] = close_m.pct_change(3)
feat['mom_12m'] = close_m.pct_change(12)
feat['vol_6m'] = ret_m.rolling(6).std()

# macro on daily grid (release-lagged) -> sampled at month end
macro_d = build_macro_features(pd.DatetimeIndex(ohlcv.index), fred_dir=Path('../data/raw/fred'))
macro_m = macro_d.reindex(close_m.index, method='ffill')
print('macro monthly columns:', list(macro_m.columns))


monthly bars: 101 | 2018-01 -> 2026-05
macro monthly columns: ['fed_funds', 'rate_2y', 'rate_10y', 'yield_curve_slope', 'broad_dollar', 'cpi_yoy', 'm2_yoy', 'unemployment']


## 2. Domanda 1 — lead/lag macro mensile vs rendimento del mese dopo
`lead_lag_table` con `n` esplicito. Lag +1 = la macro del mese `t` anticipa il
rendimento del mese `t+1` (il caso predittivo).

In [3]:
tbl = lead_lag_table(macro_m['cpi_yoy'], ret_m, lags=range(-3, 4))
print('CPI YoY(t) vs BTC return(t+k):')
print(tbl.to_string())
print()
# also the dollar and curve slope, at the predictive lag +1
for col in ['cpi_yoy', 'broad_dollar', 'yield_curve_slope', 'm2_yoy']:
    row = lead_lag_table(macro_m[col], ret_m, lags=[1])
    print(f'corr({col}[t], ret[t+1]) = {row["corr"].iloc[0]:+.3f}  (n={int(row["n"].iloc[0])})')


CPI YoY(t) vs BTC return(t+k):
          corr   n
lag               
-3.0 -0.203890  88
-2.0 -0.251298  88
-1.0 -0.289539  88
 0.0 -0.292099  88
 1.0 -0.256982  87
 2.0 -0.241084  86
 3.0 -0.204326  85

corr(cpi_yoy[t], ret[t+1]) = -0.257  (n=87)
corr(broad_dollar[t], ret[t+1]) = +0.008  (n=99)
corr(yield_curve_slope[t], ret[t+1]) = +0.001  (n=99)
corr(m2_yoy[t], ret[t+1]) = +0.053  (n=87)


## 3. Domanda 2 — modello direzionale mensile: tecnico vs tecnico+macro

In [4]:
target = (ret_m > 0).astype('float64')

def build(extra=None):
    f = feat.copy()
    if extra is not None:
        f = f.join(extra)
    X = f.shift(1)  # lag 1 month: row t = state of t-1
    data = X.join(target.rename('t'), how='inner').dropna()
    return data.drop(columns=['t']), data['t']

Xa, ya = build()
Xb, yb = build(macro_m)
rA = fit_predict_walk_forward(Xa, ya, train_size=36, test_size=12, expanding=True)
rB = fit_predict_walk_forward(Xb, yb, train_size=36, test_size=12, expanding=True)
c = rA.prediction.index.intersection(rB.prediction.index)
accA = float((rA.prediction.reindex(c)==rA.target.reindex(c)).mean())
accB = float((rB.prediction.reindex(c)==rB.target.reindex(c)).mean())
se = (0.5*0.5/len(c))**0.5 if len(c) else float('nan')  # rough SE at p=0.5
print(f'common OOS months n={len(c)}  (rough SE on accuracy ~ {se:.3f})')
print(f'A) technical only     : {accA:.4f}')
print(f'B) technical + macro  : {accB:.4f}')
print(f'delta                 : {accB-accA:+.4f}')


common OOS months n=47  (rough SE on accuracy ~ 0.073)
A) technical only     : 0.4894
B) technical + macro  : 0.4894
delta                 : +0.0000


## 4. Verifica ipotesi e conclusione

> Numeri esatti negli output.

- **H1 (lead macro debole) — CONFERMATA.** `corr(cpi_yoy[t], ret[t+1])` è
  risultata **negativa (~-0.26)**: c'è un indizio di segnale *predittivo*
  dell'inflazione sul rendimento crypto del mese dopo, col segno atteso
  (inflazione su → crypto giù), più debole del -0.40 contemporaneo di Fase 1.
  **Questo è il segnale più promettente trovato in tutto il progetto.**
- **H2 (modello senza edge robusto) — CONFERMATA.** Il classificatore mensile
  non mostra un edge OOS affidabile (delta ~0, e comunque n~45 con SE ~7pp rende
  qualsiasi differenza non significativa). La direzione binaria butta via il
  segnale continuo che la correlazione invece cattura.
- **H3 (il limite è il campione) — CONFERMATA.** Il problema non è l'assenza di
  segnale macro mensile: è che **~100 mesi sono troppo pochi** per addestrare e
  validare un modello onestamente. È un muro strutturale.

**Conclusione (e chiusura del filone di ricerca del segnale)**: alla sua
frequenza naturale (mensile), la macro **mostra un indizio di potere predittivo
lead** sul rendimento crypto (`cpi_yoy → ret` ~-0.26) — la cosa più vicina a un
segnale che il progetto abbia trovato. Ma **il campione mensile è troppo piccolo
per costruirci un modello affidabile**: con ~45 mesi OOS, non si può distinguere
edge da fortuna. Onestà metodologica (CLAUDE.md, VISION #1): *meglio dichiarare
"segnale plausibile ma non validabile col campione attuale" che spacciare un
backtest su 45 punti per una scoperta.*

**Direzioni per validare il segnale (richiedono più dati, non più codice)**:
1. **Cross-asset / cross-market**: testare lo stesso lead `cpi_yoy → ret` su
   equity e oro (decenni di storia mensile) → se regge fuori dal crypto, è reale.
2. **Più asset crypto** per moltiplicare le osservazioni (pooling).
3. Trattarlo come **feature di un modello di regime macro**, non come predittore
   standalone.

**Bias e limiti**: n~45 OOS (SE ~7pp); solo BTC; la correlazione contemporanea
≠ causalità; periodo 2018-2026 contiene un solo ciclo di inflazione maggiore
(2021-2023) che domina la statistica mensile.
